# Embeddings (continued): sentence embeddings, real APIs, and RAG

This notebook continues [`Part_4_embeddings.ipynb`](Part_4_embeddings.ipynb): where token embeddings actually come from (training a tiny word2vec from scratch), how to build an embedding for a whole *sentence*, how a real embeddings API compares to a homemade one, and why all of this matters for building better agents (RAG). It was split out of Part 4 to keep that notebook focused on token-level embeddings; it picks up exactly where Part 4 left off.

In [ ]:
# Setup: reload what we need from Part 4, since this notebook picks up where it left off.

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

## Where do embeddings come from? Training a tiny word2vec

Everything so far used embeddings already baked into a pretrained model. But where do they actually come from? The original **word2vec** approach (see [Google's Machine Learning Crash Course on embeddings](https://developers.google.com/machine-learning/crash-course/embeddings)) is a great illustration: train a small neural network on a task like predicting the next word — not because that prediction is the goal in itself, but as a *pretext task* that forces the network to organize words usefully along the way. The embeddings are simply the network's own input-lookup weights once training is done.

That's exactly the same idea as the real next-token training objective behind every LLM in this course (Part 3, Programs 7-8) — just at a toy scale, and this time actually *training* the network ourselves (with backpropagation) instead of only running inference on an already-trained one.

Let's build a tiny corpus where " king"/" queen" and " man"/" woman" always play the same grammatical role (as the subject doing something), while " dog"/" bread"/" country" always play another role (as the object something is done to) — then train a small embedding + a linear layer to predict the next word, and see what structure emerges purely from that.

In [ ]:
# Program 1: training a tiny word2vec-style embedding from scratch

import torch.nn as nn

torch.manual_seed(2)  # picked for a clean result -- see the discussion below

# A handful of sentences, repeated many times so the optimizer has enough signal.
sentences = [
    "the king rules the country",
    "the queen rules the country",
    "the man walks the dog",
    "the woman walks the dog",
    "the king is strong",
    "the queen is strong",
    "the man is strong",
    "the woman is strong",
    "the king loves the queen",
    "the man loves the woman",
    "the king eats bread",
    "the queen eats bread",
    "the man eats bread",
    "the woman eats bread",
]
corpus = " ".join(sentences * 30).split()  # simple word-level split, no BPE here

toy_vocab = sorted(set(corpus))
toy_word_to_id = {word: i for i, word in enumerate(toy_vocab)}
print(f"Toy vocabulary: {len(toy_vocab)} words -- {toy_vocab}")

# Build every (current word, next word) pair the corpus contains -- our training data.
training_pairs = [(toy_word_to_id[corpus[i]], toy_word_to_id[corpus[i + 1]]) for i in range(len(corpus) - 1)]
inputs = torch.tensor([pair[0] for pair in training_pairs])
targets = torch.tensor([pair[1] for pair in training_pairs])

# The model: an embedding table (this IS what we're actually training) followed by a
# linear layer that turns an embedding back into a score for every word in the vocabulary.
toy_embedding_dim = 8
toy_embedding = nn.Embedding(len(toy_vocab), toy_embedding_dim)
output_layer = nn.Linear(toy_embedding_dim, len(toy_vocab))

optimizer = torch.optim.Adam(list(toy_embedding.parameters()) + list(output_layer.parameters()), lr=0.03)

for epoch in range(500):
    optimizer.zero_grad()
    predicted_logits = output_layer(toy_embedding(inputs))  # look up embeddings, then predict next word
    loss = F.cross_entropy(predicted_logits, targets)  # how wrong were those predictions?
    loss.backward()  # compute how to adjust every weight, including the embeddings, to do better
    optimizer.step()

print(f"Final training loss: {loss.item():.3f}")

def toy_similarity(word_a, word_b):
    vector_a = toy_embedding.weight[toy_word_to_id[word_a]]
    vector_b = toy_embedding.weight[toy_word_to_id[word_b]]
    return F.cosine_similarity(vector_a.unsqueeze(0), vector_b.unsqueeze(0)).item()

print("\n--- 'subject' words (should end up similar to each other) ---")
for a, b in [("king", "queen"), ("man", "woman"), ("king", "man")]:
    print(f"{a:8} vs {b:8}  {toy_similarity(a, b):.3f}")

print("\n--- 'subject' vs 'object' words (should end up dissimilar) ---")
for a, b in [("king", "dog"), ("king", "country"), ("queen", "bread")]:
    print(f"{a:8} vs {b:8}  {toy_similarity(a, b):.3f}")

print("\n--- 'object' words (should also end up similar to each other) ---")
for a, b in [("dog", "bread"), ("dog", "country")]:
    print(f"{a:8} vs {b:8}  {toy_similarity(a, b):.3f}")

Structure emerges exactly where we'd hope: " king"/" queen" (0.674), " man"/" woman" (0.551) and even " king"/" man" (0.635) all end up clearly positive — these are the four words that always play the *subject* role in our sentences. Meanwhile " king" vs " dog" and " king" vs " country" are both *negative*: subjects and objects end up pushed apart. And the objects themselves (" dog", " bread", " country") cluster together too (0.755, 0.886) — sensible, since they're the words a verb like "eats" or "walks" can be followed by. (One pair, " queen"/" bread", comes out only mildly positive rather than clearly negative — a reminder that this is a *toy* example: with just 14 words and one sentence pattern, the separation isn't perfectly clean, but the overall structure is unmistakably there.)

We never told this network anything about grammar, gender, or meaning. All we asked it to do was predict the next word from the previous one — yet to get better at that narrow task, it had no choice but to organize its embedding table so that words playing the same role end up nearby, and words playing different roles end up apart. That's the whole trick behind word2vec, and behind every embedding we've inspected in this notebook: the "meaning" captured in an embedding is a byproduct of training on next-token prediction, at whatever scale — 14 words and 500 training steps here, trillions of words and a full LLM everywhere else.

## Beyond the token: embedding a whole sentence

Token embeddings are useful, but most of the time we want to compare entire sentences, not single words — for example, to find out if two sentences mean roughly the same thing. A simple, homemade way to get a sentence embedding out of any model: run the sentence through it, and average ("mean pool") the resulting per-token vectors into a single one.

In [ ]:
# Program 2: a homemade sentence embedding, by mean-pooling token vectors

from transformers import AutoModel

# AutoModel (not AutoModelForCausalLM) gives us direct access to hidden states,
# without the extra layer that predicts next-token logits.
base_model = AutoModel.from_pretrained(model_name)
base_model.eval()

def sentence_embedding(text):
    """A simple sentence embedding: the average of all its tokens' final hidden states."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        hidden_states = base_model(**inputs).last_hidden_state  # shape: (1, num_tokens, embedding_dim)
    return hidden_states.mean(dim=1).squeeze(0)  # average over the tokens -> a single vector

sentences = [
    "The cat sat on the mat.",
    "A feline was resting on the rug.",
    "The stock market crashed yesterday.",
]

embeddings = [sentence_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

The two sentences that mean roughly the same thing (the cat/feline ones) score noticeably higher than either has with the unrelated sentence about the stock market — even though they don't share a single word. That's the whole point of a sentence embedding: it captures meaning, not just vocabulary overlap.

## A real embeddings API

Mean-pooling a small local model's hidden states works, but it's a rough approximation — these models weren't specifically trained to produce good sentence embeddings. Providers instead offer dedicated **embedding models**, trained specifically to place similar meanings close together. Like everywhere else in this course, we can call one through the same OpenAI-compatible client, just changing the endpoint: `embeddings.create` instead of `chat.completions.create`. Let's use Gemini's, on the very same three sentences.

In [ ]:
# Program 3: sentence embeddings via a real embeddings API (Gemini)

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

def gemini_embedding(text):
    response = gemini.embeddings.create(model="gemini-embedding-001", input=text)
    return torch.tensor(response.data[0].embedding)

gemini_embeddings = [gemini_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(gemini_embeddings[i].unsqueeze(0), gemini_embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

Same ranking as our homemade version, but with a clearer gap between the related pair and the unrelated ones — exactly what we'd expect from a model actually trained for this task. Note also the vector length: `gemini-embedding-001` returns 3072 numbers per sentence, regardless of how long the sentence is — a fixed-size summary of its meaning.

## Why this matters for agents

<img src="images/agents.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;">

So far, every agent we've built (Part 2's `book_agent`, `recruitment_agent`, and friends) worked by stuffing *everything* it might need — the entire book, every tool description — directly into the system instructions on every single call. That's simple, but it doesn't scale: real documents can be far too large to fit in a single prompt (recall Part 3's context window), and most of that content is irrelevant to any given question anyway.

Embeddings are what makes a better approach possible: instead of sending the whole book every time, you embed it in advance (in chunks), embed the user's question the same way, and use similarity — exactly the cosine similarity we computed above — to find which chunks are actually relevant, and only send *those* to the model. This technique, retrieving relevant content by embedding similarity before generating an answer, is called **RAG** (Retrieval-Augmented Generation), and it's a natural next step for this course.

## Key takeaways

* A token id is just an integer; an **embedding** is the dense vector the model actually reasons with, one per token, stored as a row of a big matrix learned during training.
* Embeddings capture *meaning*, not spelling: tokens (or sentences) that are used similarly end up with similar vectors, measurable with **cosine similarity**.
* The same technique scales from single tokens to whole sentences (via pooling, or a dedicated embeddings API), giving a fixed-size vector that summarizes meaning regardless of length.
* This is the foundation of semantic search and **RAG**: finding relevant content by comparing embeddings, instead of stuffing everything into every prompt.